# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pratyush457/week-1-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Baseline rule

I will prioritize content pages that show a clear opportunity for improvement using signals available at the decision time.

The rule gives a higher score to pages with:
- higher search impressions, showing meaningful search visibility;
- lower click-through performance relative to their visibility, indicating a possible opportunity to improve clicks.

The rule produces one reason code: `CTR_OPPORTUNITY`.

The action label is `REVIEW_CTR`, meaning the page should be reviewed for a possible title, snippet, or search-result improvement.

This is a baseline rule, not a causal model. It is intended to create a ranked queue that can later be compared against an ML model.

In [6]:
# Section 1: Check the two signals used by the baseline rule

%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

base = "hf://datasets/FlyRank/internship-warehouse"

march = f"""
read_parquet(
    '{base}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# Signal 1: Search visibility / impressions
impression_buckets = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions < 100 THEN '<100'
            WHEN gsc_impressions < 500 THEN '100-499'
            WHEN gsc_impressions < 1000 THEN '500-999'
            ELSE '1000+'
        END AS impressions_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks
    FROM {march}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
    GROUP BY 1
    ORDER BY
        CASE impressions_bucket
            WHEN '<100' THEN 1
            WHEN '100-499' THEN 2
            WHEN '500-999' THEN 3
            ELSE 4
        END
""").df()

print("SIGNAL 1 — GSC IMPRESSIONS")
print(impression_buckets.to_string(index=False))


# Signal 2: Click-through rate
ctr_buckets = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions = 0 THEN 'No impressions'
            WHEN 100.0 * gsc_clicks / gsc_impressions < 1 THEN '<1%'
            WHEN 100.0 * gsc_clicks / gsc_impressions < 3 THEN '1-2.99%'
            WHEN 100.0 * gsc_clicks / gsc_impressions < 5 THEN '3-4.99%'
            ELSE '5%+'
        END AS ctr_bucket,
        COUNT(*) AS n,
        ROUND(
            AVG(
                CASE
                    WHEN gsc_impressions > 0
                    THEN 100.0 * gsc_clicks / gsc_impressions
                END
            ), 2
        ) AS avg_ctr_pct
    FROM {march}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
    GROUP BY 1
    ORDER BY
        CASE ctr_bucket
            WHEN 'No impressions' THEN 1
            WHEN '<1%' THEN 2
            WHEN '1-2.99%' THEN 3
            WHEN '3-4.99%' THEN 4
            ELSE 5
        END
""").df()

print("\nSIGNAL 2 — CTR")
print(ctr_buckets.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1 — GSC IMPRESSIONS
impressions_bucket       n  avg_clicks
              <100 2972453        0.06
           100-499  537157        0.65
           500-999   69032        1.94
             1000+   32419        5.07

SIGNAL 2 — CTR
ctr_bucket       n  avg_ctr_pct
       <1% 3413195         0.03
   1-2.99%  133370         1.69
   3-4.99%   28006         3.77
       5%+   36490        18.39


### Signal verdicts

**1. GSC impressions — CONFIRMED**

The bucket table shows a clear increase in average clicks as impressions increase: pages with fewer than 100 impressions average 0.06 clicks, while pages with 1,000+ impressions average 5.07 clicks. This supports using search visibility/volume as a baseline prioritization signal. This is linked to the FlyRank quick-win logic, where search volume is used to identify opportunities.

**2. CTR — CONFIRMED**

The CTR buckets show a strong relationship between CTR and click performance. The lowest bucket (<1%) has an average CTR of 0.03%, while the 5%+ bucket has an average CTR of 18.39%. This supports using CTR as a signal for identifying pages that may deserve search-result improvement review. This is linked to the FlyRank CTR-fix logic.

Both signals are measured from March data and are available at the decision time; no future-window or label-derived data is used.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Section 2: Build the baseline ranked action queue

import os
import pandas as pd

queue = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE NULL
        END AS ctr_pct
    FROM {march}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_impressions > 0
""").df()

# Higher impressions + lower CTR = higher baseline opportunity score
queue["score"] = (
    __import__("numpy").log1p(queue["gsc_impressions"])
    * (1 - queue["ctr_pct"].clip(lower=0, upper=100) / 100)
)

# One reason code and one action label
queue["reason_code"] = "CTR_OPPORTUNITY"
queue["action"] = "REVIEW_CTR"

# Rank highest-priority opportunities first
queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# Keep the output compact and explicit
queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "score",
        "reason_code",
        "action"
    ]
]

# Write the required CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Rows in ranked queue:", len(queue))
print("CSV written to:", output_path)

queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in ranked queue: 3611061
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr_pct,score,reason_code,action
0,1,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-28,40084,1,0.002495,10.598493,CTR_OPPORTUNITY,REVIEW_CTR
1,2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-04,39003,2,0.005128,10.570877,CTR_OPPORTUNITY,REVIEW_CTR
2,3,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.000000,10.528597,CTR_OPPORTUNITY,REVIEW_CTR
3,4,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-29,39305,252,0.641140,10.511305,CTR_OPPORTUNITY,REVIEW_CTR
4,5,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-28,38436,271,0.705068,10.482343,CTR_OPPORTUNITY,REVIEW_CTR
5,6,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.000000,10.415832,CTR_OPPORTUNITY,REVIEW_CTR
6,7,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-30,35404,225,0.635521,10.408040,CTR_OPPORTUNITY,REVIEW_CTR
7,8,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-27,32958,0,0.000000,10.403020,CTR_OPPORTUNITY,REVIEW_CTR
8,9,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-29,32756,2,0.006106,10.396237,CTR_OPPORTUNITY,REVIEW_CTR
9,10,client_e547b89c05043229,content_eadb33b5df496f4a,2026-03-27,34817,223,0.640492,10.390908,CTR_OPPORTUNITY,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

The top 20 rows are reviewed using only the March decision-time signals used by the baseline rule.

Each review records:
- the recommended action;
- the single reason code produced by the rule;
- a confidence note based on the observed signal strength;
- what evidence would make the recommendation wrong.

These are prioritization recommendations, not causal conclusions. A page should be manually reviewed before any change is made.

In [8]:
# Section 3: Top-20 review

top20 = queue.head(20).copy()

def confidence_note(row):
    if row["gsc_impressions"] >= 1000 and row["ctr_pct"] < 1:
        return "High: strong visibility with very low CTR."
    elif row["gsc_impressions"] >= 500 and row["ctr_pct"] < 3:
        return "Medium: meaningful visibility with below-3% CTR."
    else:
        return "Lower: signal is weaker and needs manual review."

def wrong_if(row):
    return (
        "Wrong if the low CTR is appropriate for the page's search intent, "
        "or if the page is already optimized and no CTR improvement is realistic."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print("Top-20 review:")
display(review)

Top-20 review:


,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_44f34c0a90047651,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
1,2,content_34a70fea29d15f24,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
2,3,content_945d6ff91386c817,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
3,4,content_eadb33b5df496f4a,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
4,5,content_eadb33b5df496f4a,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
5,6,content_fec55986a1868d62,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
6,7,content_eadb33b5df496f4a,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
7,8,content_44f34c0a90047651,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
8,9,content_44f34c0a90047651,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...
9,10,content_eadb33b5df496f4a,REVIEW_CTR,CTR_OPPORTUNITY,High: strong visibility with very low CTR.,Wrong if the low CTR is appropriate for the pa...


## 4. Weak picks + leakage check

Some top picks could be weak recommendations even when their score is high.

A pick could be wrong if its low CTR is expected for the page's search intent, if the search result contains features that reduce clicks, or if the page is already performing well enough that a CTR change is not a realistic opportunity. This is especially relevant for pages that have high impressions but already receive substantial clicks.

The baseline uses only March decision-time fields: `gsc_impressions`, `gsc_clicks`, and the CTR calculated from them. No future-window performance fields, future labels, or product flags are used as inputs to the score.

The queue is therefore a prioritization baseline, not proof that every selected page needs a CTR change.

In [9]:
# Leakage and forbidden-input check

allowed_inputs = {
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct"
}

forbidden_terms = [
    "future",
    "label",
    "product_flag",
    "refresh_flag",
    "outcome",
    "april",
    "may",
    "june"
]

used_inputs = {"gsc_impressions", "gsc_clicks", "ctr_pct"}

leaked_inputs = [
    col for col in used_inputs
    if any(term in col.lower() for term in forbidden_terms)
]

print("Inputs used by baseline:", sorted(used_inputs))
print("Forbidden/future inputs found:", leaked_inputs)

assert not leaked_inputs, "Leakage detected!"

print("Leakage check: PASSED")

Inputs used by baseline: ['ctr_pct', 'gsc_clicks', 'gsc_impressions']
Forbidden/future inputs found: []
Leakage check: PASSED


## Self-check

Before submitting, I confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.
- [x] Two decision-time signals were checked with visible bucket tables, `n`, and verdicts.
- [x] The baseline has one score, one reason code, and one action label.
- [x] The ranked queue was written to `work/outputs/baseline_action_score.csv`.
- [x] The top-20 review includes action, reason code, confidence note, and what would make each pick wrong.
- [x] Weak picks and leakage were reviewed; the leakage check passed.